## What is Zero-Shot Learning?

- <span style="color:green"> **Zero-Shot Learning** </span> is a concept, that a model when trained on enough unlabeled data (unsupervised learning) is able to generalize/ recognize at inference time even though the model was not trained on the inference data. This can be used in NLP, Images etc.
- <span style="color:green"> **Zero-Shot Learning** </span> is a setup in which a model can learn to recognize things that it hasn’t explicitly seen before in training.

#### Load useful libraries and data

In [1]:
from transformers import pipeline
import pandas as pd
import numpy as np
from tqdm import tqdm


In [2]:
data = pd.read_csv(
    "data/SMSSpamCollection.txt",
    encoding="utf-8",
    header=None,
    delimiter="\t",
    names=["target", "text"],
)

In [3]:
data.head(5)

,target,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
print(f'There are {data.shape[0]} rows in the dataset')

There are 5572 rows in the dataset


### Preparing the pipeline in one-line of code!

In [5]:
classifier = pipeline("zero-shot-classification",device = 0)

No model was supplied, defaulted to FacebookAI/roberta-large-mnli and revision 130fb28 (https://huggingface.co/FacebookAI/roberta-large-mnli).
Using a pipeline without specifying a model name and revision in production is not recommended.
c:\Users\thngu\neuefische_course\pepperprompt\fork-ds-intro-to-NLP\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

c:\Users\thngu\neuefische_course\pepperprompt\fork-ds-intro-to-NLP\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\thngu\.cache\huggingface\hub\models--FacebookAI--roberta-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.43G [00:00<?, ?B/s]

All PyTorch model weights were used when initializing TFRobertaForSequenceClassification.

All the weights of TFRobertaForSequenceClassification were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFRobertaForSequenceClassification for predictions without further training.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

### Making Predictions

This model works best with informative labels, spam/ham are not so informative. Using spam/ham leads to a Hamming loss of 53% vs using click bait/written by humans leading to 19%

Can you find better label descriptions?

In [6]:
category_map = {"spam":"click bait", "ham":"written by humans"}

In [7]:
candidate_labels = list(category_map.values())
predictedCategories = []
trueCategories = []
for i in tqdm(range(100)):
    text = data.iloc[i,]['text']
    cat = [data.iloc[i,]['target']]
    res = classifier(text, candidate_labels, multi_label=False)
    labels = res['labels'] 
    scores = res['scores'] #extracting the scores associated with the labels
    res_dict = {label : score for label,score in zip(labels, scores)}
    sorted_dict = dict(sorted(res_dict.items(), key=lambda x:x[1],reverse = True)) #sorting the dictionary of labels in descending order based on their score
    categories  = next(k for i, (k,v) in enumerate(sorted_dict.items()))

    predictedCategories.append(categories)
    trueCats = [category_map[x] for x in cat]
    trueCategories.append(trueCats)

100%|██████████| 100/100 [02:27<00:00,  1.48s/it]


In [8]:
for y_true, y_pred in zip(trueCategories[:3], predictedCategories[:3]):
    print(f'True Categories {y_true}')
    print(f'Predicted Categories {y_pred}')
    print('#'*50)

True Categories ['written by humans']
Predicted Categories written by humans
##################################################
True Categories ['written by humans']
Predicted Categories written by humans
##################################################
True Categories ['click bait']
Predicted Categories written by humans
##################################################


### Hamming Loss
The Hamming loss is the fraction of labels that are incorrectly predicted.

In [9]:
from sklearn.metrics import hamming_loss
print(f'The hamming loss is {hamming_loss(trueCategories,predictedCategories):.4f} compared to 0.0237 from the last trained model in Notebook 1')

The hamming loss is 0.1900 compared to 0.0237 from the last trained model in Notebook 1


# This Jupyter Notebook snippet is a goldmine for an AI Project Manager. It perfectly illustrates one of the most cost-effective strategies in modern AI: **Label Engineering** (a subset of Prompt Engineering).

Here is the PM perspective on what is happening in this code, why it matters, and how you can solve the challenge at the end of the notebook.

### The Problem: Why does "Ham" fail?

In traditional machine learning, "Spam" and "Ham" were just arbitrary tags (like `0` and `1`). The model didn't know what the words meant; it just learned that messages with the word "FREE" usually mapped to the `0` tag.

But as we discussed earlier, **Zero-Shot Learning** relies on **Contextual Embeddings**. The model uses the *actual dictionary meaning* of the label to classify the text.

* To a Zero-Shot model, "Spam" means unsolicited junk. (Good).
* To a Zero-Shot model, "Ham" means a delicious cut of pork. (Bad).

When the model reads a text saying *"Hey, are we still on for dinner?"*, it compares the mathematical meaning of that sentence to "Spam" and "Pork". It gets horribly confused, resulting in a terrible **Hamming Loss** (the percentage of times the model guessed completely wrong).

### The PM Solution: Label Engineering

As an AI PM, you don't need to ask your Data Scientists to spend thousands of dollars fine-tuning a model on a new dataset. You just need to change the English words in the `category_map` dictionary to be more semantically descriptive.

Here are the "better labels" the notebook is asking for to improve your Hamming Loss:

**Better Labels for "Spam":**

* *"Promotional advertisement"*
* *"Automated marketing message"*
* *"Malicious phishing scam"*
* *"Unsolicited financial offer"*

**Better Labels for "Ham":**

* *"Personal conversation between friends"*
* *"Direct human communication"*
* *"Casual text message"*
* *"Standard conversational reply"*

By simply changing the labels to *"Automated marketing"* vs. *"Personal conversation"*, the Zero-Shot model immediately understands the assignment because those phrases have massive semantic footprints in its embedding space.

---

### Interactive Concept: The Zero-Shot Label Simulator

To see exactly how much your choice of words impacts the AI's confidence, I have built a **Zero-Shot Simulator** below.

Try testing the terrible "Spam vs. Ham" labels against a standard text message, and then try swapping them out for the descriptive labels I listed above to see how the "AI" mathematically shifts its confidence!

## Check your understanding

Can you find better "labels" for the category description improve the Hamming Loss?

In [10]:
!pip install gradio

  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
   ---------------------------------------- 0.0/32.3 MB ? eta -:--:--
   - -------------------------------------- 1.3/32.3 MB 6.7 MB/s eta 0:00:05
   --- ------------------------------------ 2.6/32.3 MB 6.9 MB/s eta 0:00:05
   ---- ----------------------------------- 3.9/32.3 MB 6.7 MB/s eta 0:00:05
   ------ --------------------------------- 5.5/32.3 MB 6.7 MB/s eta 0:00:04
   -------- ------------------------------- 6.6/32.3 MB 6.7 MB/s eta 0:00:04
   ---------- ----------------------------- 8.1/32.3 MB 6.7 MB/s eta 0:00:04
   ------------ --------------------------- 9.7/32.3 MB 6.7 MB/s eta 0:00:04
   ------------- -------------------------- 11.0/32.3 MB 6.7 MB/s eta 0:00:04
   --------------- -------

In [12]:
import gradio as gr

# This function uses the 'classifier' pipeline you already loaded in your notebook
def simulate_zero_shot(message, label_1, label_2):
    candidate_labels = [label_1, label_2]
    
    # Run the model
    res = classifier(message, candidate_labels, multi_label=False)
    
    # Format the output so Gradio can build a nice percentage bar chart
    return {label: float(score) for label, score in zip(res['labels'], res['scores'])}

# Build the interactive UI
demo = gr.Interface(
    fn=simulate_zero_shot,
    inputs=[
        gr.Dropdown(
            choices=[
                "URGENT! You have won a 1 week FREE membership! Txt CLAIM to 81010", 
                "Hey man, are we still on for poker tonight? Bring snacks."
            ], 
            label="Select an SMS Message",
            allow_custom_value=True # Lets you type your own message too!
        ),
        gr.Textbox(label="Label 1 (Try: 'Spam' vs 'Promotional Scam')", value="Spam"),
        gr.Textbox(label="Label 2 (Try: 'Ham' vs 'Personal Conversation')", value="Ham")
    ],
    outputs=gr.Label(label="Model Confidence Split"),
    title="PM Prototype: Zero-Shot Label Engineering",
    description="Test how changing vague labels to descriptive semantics impacts the model's confidence."
)

# Launch the UI directly inside the Jupyter Notebook
demo.launch(inline=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [11]:
import gradio as gr

# This function uses the 'classifier' pipeline you already loaded in your notebook
def simulate_zero_shot(message, label_1, label_2):
    candidate_labels = [label_1, label_2]
    
    # Run the model
    res = classifier(message, candidate_labels, multi_label=False)
    
    # Format the output so Gradio can build a nice percentage bar chart
    return {label: float(score) for label, score in zip(res['labels'], res['scores'])}

# Build the interactive UI
demo = gr.Interface(
    fn=simulate_zero_shot,
    inputs=[
        gr.Dropdown(
            choices=[
                "URGENT! You have won a 1 week FREE membership! Txt CLAIM to 81010", 
                "Hey man, are we still on for poker tonight? Bring snacks."
            ], 
            label="Select an SMS Message",
            allow_custom_value=True # Lets you type your own message too!
        ),
        gr.Textbox(label="Label 1 (Try: 'Spam' vs 'Promotional Scam')", value="Spam"),
        gr.Textbox(label="Label 2 (Try: 'Ham' vs 'Personal Conversation')", value="Ham")
    ],
    outputs=gr.Label(label="Model Confidence Split"),
    title="PM Prototype: Zero-Shot Label Engineering",
    description="Test how changing vague labels to descriptive semantics impacts the model's confidence."
)

# Launch the UI directly inside the Jupyter Notebook
demo.launch(inline=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [13]:
import gradio as gr

# This function uses the 'classifier' pipeline you already loaded in your notebook
def simulate_zero_shot(message, label_1, label_2):
    candidate_labels = [label_1, label_2]
    
    # Run the model
    res = classifier(message, candidate_labels, multi_label=False)
    
    # Format the output so Gradio can build a nice percentage bar chart
    return {label: float(score) for label, score in zip(res['labels'], res['scores'])}

# Build the interactive UI
demo = gr.Interface(
    fn=simulate_zero_shot,
    inputs=[
        gr.Dropdown(
            choices=[
                "URGENT! You have won a 1 week FREE membership! Txt CLAIM to 81010", 
                "Hey man, are we still on for poker tonight? Bring snacks."
            ], 
            label="Select an SMS Message",
            allow_custom_value=True # Lets you type your own message too!
        ),
        gr.Textbox(label="Label 1 (Try: 'Spam' vs 'Promotional Scam')", value="Spam"),
        gr.Textbox(label="Label 2 (Try: 'Ham' vs 'Personal Conversation')", value="Ham")
    ],
    outputs=gr.Label(label="Model Confidence Split"),
    title="PM Prototype: Zero-Shot Label Engineering",
    description="Test how changing vague labels to descriptive semantics impacts the model's confidence."
)

# Launch the UI directly inside the Jupyter Notebook
demo.launch(inline=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
